# BEHRT MLM Pre-training - QUICK VERSION (30-60 minutes)

**This notebook is optimized for fast training:**
- Smaller model (hidden_size: 144 vs 288)
- Fewer layers (3 vs 6)
- Smaller batch size (128 vs 256)
- Fewer epochs (10 vs 50)
- Uses subset of data (30%)

**Use this for:**
- Quick testing and prototyping
- Learning how the model works
- Limited compute resources

**For production/research, use MLM.ipynb (full version)**

In [53]:
import sys 
sys.path.insert(0, '../')

In [54]:
from common.common import create_folder
from common.pytorch import load_model
import pytorch_pretrained_bert as Bert
from model.utils import age_vocab
from common.common import load_obj
from dataLoader.MLM import MLMLoader
from torch.utils.data import DataLoader
import pandas as pd
from model.MLM import BertForMaskedLM
from model.optimiser import adam
import sklearn.metrics as skm
import numpy as np
import torch
import time
import torch.nn as nn
import os

In [55]:
class BertConfig(Bert.modeling.BertConfig):
    def __init__(self, config):
        super(BertConfig, self).__init__(
            vocab_size_or_config_json_file=config.get('vocab_size'),
            hidden_size=config['hidden_size'],
            num_hidden_layers=config.get('num_hidden_layers'),
            num_attention_heads=config.get('num_attention_heads'),
            intermediate_size=config.get('intermediate_size'),
            hidden_act=config.get('hidden_act'),
            hidden_dropout_prob=config.get('hidden_dropout_prob'),
            attention_probs_dropout_prob=config.get('attention_probs_dropout_prob'),
            max_position_embeddings = config.get('max_position_embedding'),
            initializer_range=config.get('initializer_range'),
        )
        self.seg_vocab_size = config.get('seg_vocab_size')
        self.age_vocab_size = config.get('age_vocab_size')
        
class TrainConfig(object):
    def __init__(self, config):
        self.batch_size = config.get('batch_size')
        self.use_cuda = config.get('use_cuda')
        self.max_len_seq = config.get('max_len_seq')
        self.train_loader_workers = config.get('train_loader_workers')
        self.test_loader_workers = config.get('test_loader_workers')
        self.device = config.get('device')
        self.output_dir = config.get('output_dir')
        self.output_name = config.get('output_name')
        self.best_name = config.get('best_name')

In [56]:
# ============ QUICK VERSION CONFIG ============
file_config = {
    'vocab': '../data/processed/vocab_ccsr',  # CCSR vocab
    'data': '../data/processed/train_mlm_ccsr.parquet',  # CCSR data
    'model_path': '../data/models/quick/',
    'model_name': 'behrt_mlm_ccsr_quick.pt',  # CCSR model
    'file_name': 'mlm_training_ccsr_quick.log',
}
create_folder(file_config['model_path'])

In [57]:
# ============ REDUCED PARAMETERS FOR SPEED ============
global_params = {
    'max_seq_len': 64,  # Reduced from 64
    'max_age': 110,
    'month': 1,
    'age_symbol': None,
    'min_visit': 5,
    'gradient_accumulation_steps': 1,
    'data_fraction': 0.5  # Use only 30% of data for quick training
}

optim_param = {
    'lr': 5e-5,  # Slightly higher learning rate for faster convergence
    'warmup_proportion': 0.1,
    'weight_decay': 0.01
}

train_params = {
    'batch_size': 256,  # Reduced from 256
    'use_cuda': False,
    'max_len_seq': global_params['max_seq_len'],
    'device': 'cpu',
}

In [58]:
BertVocab = load_obj(file_config['vocab'])
ageVocab, _ = age_vocab(max_age=global_params['max_age'], mon=global_params['month'], symbol=global_params['age_symbol'])

In [59]:
# Load data and use only a fraction for quick training
data = pd.read_parquet(file_config['data'])
print(f"Original data size: {len(data)}")

# Use only a fraction of data
data = data.sample(frac=global_params['data_fraction'], random_state=42)
print(f"Quick training data size: {len(data)} ({global_params['data_fraction']*100}% of original)")

# remove patients with visits less than min visit
data['length'] = data['code'].apply(lambda x: len([i for i in range(len(x)) if x[i] == 'SEP']))
data = data[data['length'] >= global_params['min_visit']]
data = data.reset_index(drop=True)
print(f"After filtering (min {global_params['min_visit']} visits): {len(data)}")

Original data size: 19783
Quick training data size: 9892 (50.0% of original)
After filtering (min 5 visits): 7211


In [60]:
Dset = MLMLoader(data, BertVocab['token2idx'], ageVocab, max_len=train_params['max_len_seq'], code='code')
trainload = DataLoader(dataset=Dset, batch_size=train_params['batch_size'], shuffle=True, num_workers=3)
data_len = len(trainload)

In [61]:
# ============ SMALLER MODEL CONFIGURATION ============
model_config = {
    'vocab_size': len(BertVocab['token2idx'].keys()),
    'hidden_size': 144,  # REDUCED from 288 (2x faster)
    'seg_vocab_size': 2,
    'age_vocab_size': len(ageVocab.keys()),
    'max_position_embedding': train_params['max_len_seq'],
    'hidden_dropout_prob': 0.1,
    'num_hidden_layers': 3,  # REDUCED from 6 (2x faster)
    'num_attention_heads': 6,  # REDUCED from 12
    'attention_probs_dropout_prob': 0.1,
    'intermediate_size': 256,  # REDUCED from 512
    'hidden_act': 'gelu',
    'initializer_range': 0.02,
}

print("\n=" * 60)
print("QUICK TRAINING CONFIGURATION")
print("=" * 60)
print(f"Model size: ~30% of full model")
print(f"Hidden size: {model_config['hidden_size']} (vs 288 full)")
print(f"Layers: {model_config['num_hidden_layers']} (vs 6 full)")
print(f"Attention heads: {model_config['num_attention_heads']} (vs 12 full)")
print(f"Batch size: {train_params['batch_size']} (vs 256 full)")
print(f"Data fraction: {global_params['data_fraction']*100}%")
print(f"Expected training time: 30-60 minutes")
print("=" * 60)


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
QUICK TRAINING CONFIGURATION
Model size: ~30% of full model
Hidden size: 144 (vs 288 full)
Layers: 3 (vs 6 full)
Attention heads: 6 (vs 12 full)
Batch size: 256 (vs 256 full)
Data fraction: 50.0%
Expected training time: 30-60 minutes


In [62]:
conf = BertConfig(model_config)
model = BertForMaskedLM(conf)

In [63]:
model = model.to(train_params['device'])
optim = adam(params=list(model.named_parameters()), config=optim_param)

t_total value of -1 results in schedule not being applied


In [64]:
def cal_acc(label, pred):
    logs = nn.LogSoftmax()
    label=label.cpu().numpy()
    ind = np.where(label!=-1)[0]
    truepred = pred.detach().cpu().numpy()
    truepred = truepred[ind]
    truelabel = label[ind]
    truepred = logs(torch.tensor(truepred))
    outs = [np.argmax(pred_x) for pred_x in truepred.numpy()]
    precision = skm.precision_score(truelabel, outs, average='micro')
    return precision

In [65]:
def train(e, loader):
    tr_loss = 0
    temp_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0
    cnt= 0
    start = time.time()

    for step, batch in enumerate(loader):
        cnt +=1
        batch = tuple(t.to(train_params['device']) for t in batch)
        age_ids, input_ids, posi_ids, segment_ids, attMask, masked_label = batch
        loss, pred, label = model(input_ids, age_ids, segment_ids, posi_ids,attention_mask=attMask, masked_lm_labels=masked_label)
        if global_params['gradient_accumulation_steps'] >1:
            loss = loss/global_params['gradient_accumulation_steps']
        loss.backward()
        
        temp_loss += loss.item()
        tr_loss += loss.item()
        
        nb_tr_examples += input_ids.size(0)
        nb_tr_steps += 1
        
        # Print every 50 steps (more frequent for quick training)
        if step % 50 == 0:
            print("epoch: {}\t| step: {}/{}\t| Loss: {:.4f}\t| precision: {:.4f}\t| time: {:.2f}s".format(
                e, step, len(loader), temp_loss/(step+1), cal_acc(label, pred), time.time()-start))
            
        if (step + 1) % global_params['gradient_accumulation_steps'] == 0:
            optim.step()
            optim.zero_grad()

    print("\n** ** * Saving model ** ** * ")
    model_to_save = model.module if hasattr(model, 'module') else model
    create_folder(file_config['model_path'])
    output_model_file = os.path.join(file_config['model_path'], file_config['model_name'])
    torch.save(model_to_save.state_dict(), output_model_file)
    print(f"Model saved to: {output_model_file}\n")
        
    cost = time.time() - start
    return tr_loss, cost

In [66]:
# ============ QUICK TRAINING: 10 EPOCHS (vs 50 full) ============
f = open(os.path.join(file_config['model_path'], file_config['file_name']), "w")
f.write('{}\t{}\t{}\n'.format('epoch', 'loss', 'time'))

print("\n" + "="*60)
print("STARTING QUICK TRAINING - 10 EPOCHS")
print("="*60)
print("Expected time: 30-60 minutes")
print("="*60 + "\n")

total_start = time.time()

for e in range(10):  # Only 10 epochs instead of 50
    print(f"\n{'='*60}")
    print(f"EPOCH {e+1}/10")
    print(f"{'='*60}")
    loss, time_cost = train(e, trainload)
    loss = loss/data_len
    f.write('{}\t{}\t{}\n'.format(e, loss, time_cost))
    print(f"Epoch {e+1} complete - Loss: {loss:.4f} - Time: {time_cost/60:.2f} min")

f.close()

total_time = (time.time() - total_start) / 60
print("\n" + "="*60)
print(f"QUICK TRAINING COMPLETE!")
print("="*60)
print(f"Total training time: {total_time:.2f} minutes")
print(f"Model saved to: {file_config['model_path']}{file_config['model_name']}")
print(f"Training log: {file_config['model_path']}{file_config['file_name']}")
print("\nNext step: Run NextXVisit_QUICK.ipynb for fine-tuning")
print("="*60)


STARTING QUICK TRAINING - 10 EPOCHS
Expected time: 30-60 minutes


EPOCH 1/10


/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


epoch: 0	| step: 0/29	| Loss: 6.1938	| precision: 0.0008	| time: 5.58s

** ** * Saving model ** ** * 
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt

Epoch 1 complete - Loss: 5.5154 - Time: 0.66 min

EPOCH 2/10


/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


epoch: 1	| step: 0/29	| Loss: 5.0160	| precision: 0.0870	| time: 3.43s

** ** * Saving model ** ** * 
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt

Epoch 2 complete - Loss: 4.8413 - Time: 0.61 min

EPOCH 3/10


...

epoch: 7	| step: 0/29	| Loss: 4.5556	| precision: 0.1217	| time: 3.36s

** ** * Saving model ** ** * 
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt

Epoch 8 complete - Loss: 4.5423 - Time: 0.57 min

EPOCH 9/10


/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


epoch: 8	| step: 0/29	| Loss: 4.4516	| precision: 0.1284	| time: 3.30s

** ** * Saving model ** ** * 
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt

Epoch 9 complete - Loss: 4.4960 - Time: 0.56 min

EPOCH 10/10


/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


epoch: 9	| step: 0/29	| Loss: 4.4829	| precision: 0.1179	| time: 3.24s

** ** * Saving model ** ** * 
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt

Epoch 10 complete - Loss: 4.4548 - Time: 0.57 min

QUICK TRAINING COMPLETE!
Total training time: 5.95 minutes
Model saved to: ../data/models/quick/behrt_mlm_ccsr_quick.pt
Training log: ../data/models/quick/mlm_training_ccsr_quick.log

Next step: Run NextXVisit_QUICK.ipynb for fine-tuning
